# Belief State Basics

Introduction to POMDP belief states and their quantum encoding.

Key concepts:
- Belief as probability distribution over states
- Classical Bayesian update
- Quantum amplitude encoding
- Distance metrics (KL, Hellinger, fidelity)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantum_common.visualization.styles import apply_publication_style

apply_publication_style()

from quantum_pomdp.models.belief_state import BeliefState
from quantum_pomdp.scenarios.tiger_problem import create_tiger_pomdp

In [ ]:
# Create Tiger POMDP
model = create_tiger_pomdp(listen_accuracy=0.85)
print(f"States: {model.num_states}, Actions: {model.num_actions}, Observations: {model.num_observations}")

# Uniform belief
b = BeliefState.uniform(2)
print(f"\nUniform belief: {b.probabilities}")
print(f"Entropy: {b.entropy():.4f} (max = {np.log2(2):.4f})")
print(f"Most likely state: {b.most_likely_state}")

In [ ]:
# Classical Bayesian belief update
beliefs = [b.probabilities.copy()]
current = b

# Simulate 10 'listen' actions with 'hear-left' observations
for step in range(10):
    current = current.classical_update(
        action=0, observation=0,  # listen, hear-left
        transition_tensor=model.transition_tensor,
        observation_tensor=model.observation_tensor,
    )
    beliefs.append(current.probabilities.copy())

beliefs = np.array(beliefs)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(beliefs[:, 0], 'o-', label='P(tiger-left)')
ax.plot(beliefs[:, 1], 's-', label='P(tiger-right)')
ax.set_xlabel('Step')
ax.set_ylabel('Probability')
ax.set_title('Belief Update: Listening with hear-left observations')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Quantum amplitude encoding
b_skewed = BeliefState(np.array([0.8, 0.2]))
amplitudes = b_skewed.to_amplitudes()
print(f"Belief: {b_skewed.probabilities}")
print(f"Amplitudes: {amplitudes}")
print(f"Verify |a|^2: {np.abs(amplitudes)**2}")

In [ ]:
# Distance metrics
b1 = BeliefState(np.array([0.9, 0.1]))
b2 = BeliefState(np.array([0.5, 0.5]))

print(f"KL divergence: {b1.kl_divergence(b2):.4f}")
print(f"Hellinger distance: {b1.hellinger_distance(b2):.4f}")
print(f"Total variation: {b1.total_variation_distance(b2):.4f}")
print(f"Fidelity: {b1.fidelity(b2):.4f}")